In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.feature import NaturalEarthFeature
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.cm import get_cmap
import metpy.calc as mpcalc
from metpy.units import units
from numpy import *
import xarray as xr
from netCDF4 import Dataset, num2date
import math
import pygrib
import cdsapi
from datetime import datetime, timedelta

In [ ]:
############# Declare time, level, and lat/lon boundaries here ##########
startdate = '12/22/13 12:00'    ## must be in this format 'MM/DD/YY HH:MM'
latN = 90
latS = 0
lonW = -180    # must be in degrees E (The western hemisphere is captured between -180 and 0 degrees east)
lonE = 0
date = datetime.strptime(startdate,'%m/%d/%y %H:%M')
level= 850

In [ ]:
############### ONLY RUN THIS CELL ONCE TO ACQUIRE DATA #################
############### ONCE YOU HAVE RUN THIS CELL AND DOWNLOADED THE DATA YOU SHOULD COMMENT OUT ALL OF THE LINES BY PUTTING A # IN FRONT OF EVERY LINE #################
############### To see all of the possible data you can download from ERA5 go to the websites below. If you click "Show API Request" on the bottom of the page it will give you the info you can substitute in the request below for your own purposes 
############### Data on pressure levels: https://cds-beta.climate.copernicus.eu/datasets/reanalysis-era5-pressure-levels?tab=download
############### Data on single levels: https://cds-beta.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download

filestr = 'synoptic_'+date.strftime('%Y')+date.strftime('%m')+date.strftime('%d')+'_'+date.strftime('%H')+'00.nc'     # you can change these filenames as you see fit
print(filestr)      

######### Download ERA5 data using CDS-API ##########
client = cdsapi.Client()
dataset = "reanalysis-era5-pressure-levels"
request = {
    'product_type': ['reanalysis'],
    'variable': ['geopotential','temperature','u_component_of_wind','v_component_of_wind','vertical_velocity'],
    'year': [date.strftime('%Y')],
    'month': [date.strftime('%m')],
    'day': [date.strftime('%d')],
    'time': [date.strftime('%H:%M')],
    'pressure_level': ['850'],
    'data_format': 'netcdf',
    'download_format': 'unarchived',
    'area': [latN, lonW, latS, lonE]
}
client.retrieve(dataset, request, filestr)

In [ ]:
######## Enter in the datasets #########
ds = xr.open_dataset(filestr).metpy.parse_cf()

######## Read in the variables ##########
uwnd = ds.u.metpy.sel(valid_time=date,pressure_level=level,latitude=slice(latN,latS),longitude=slice(lonW,lonE))
lats = ds.latitude.metpy.sel(latitude=slice(latN,latS))
lons = ds.longitude.metpy.sel(longitude=slice(lonW,lonE))
lons_2D, lats_2D = meshgrid(lons,lats)

vwnd = ds.v.metpy.sel(valid_time=date,pressure_level=level,latitude=slice(latN,latS),longitude=slice(lonW,lonE))
geop = ds.z.metpy.sel(valid_time=date,pressure_level=level,latitude=slice(latN,latS),longitude=slice(lonW,lonE))
tmpk = ds.t.metpy.sel(valid_time=date,pressure_level=level,latitude=slice(latN,latS),longitude=slice(lonW,lonE))
dx, dy = mpcalc.lat_lon_grid_deltas(lons, lats)
hght = geop/9.81    # need to divide the geopotential by gravity to get the geopotential height

corl = 2.0*7.292e-5*sin(lats*(pi/180.)) / units.second     # Coriolis Parameter

In [ ]:
######### Apply a spatial smoother to the variables so that synoptic-scale signals can be more readily observed  ##########

######### A weight of "25" is used below, which is typically good for picking out synoptic-scale features. You can experiment with changing these weights to see the difference ##########
hght = mpcalc.smooth_gaussian(hght,25)
uwnd = mpcalc.smooth_gaussian(uwnd,25)
vwnd = mpcalc.smooth_gaussian(vwnd,25)
tmpk = mpcalc.smooth_gaussian(tmpk,25)

In [ ]:
########### Calculate additional variables to plot #########
theta = mpcalc.potential_temperature(level * units.hPa, tmpk)

In [ ]:
######## Create the plot #######

##### Set up the projection that will be used ##########
mapcrs = ccrs.LambertConformal(central_longitude=-55, central_latitude=45,
                               standard_parallels=(33, 45))
mapcrs = ccrs.PlateCarree()

# Set up the projection of the data; if lat/lon then PlateCarree is what you want
datacrs = ccrs.PlateCarree()

# Start the figure and create plot axes with proper projection
fig = plt.figure(1, figsize=(14, 12))
ax = plt.subplot(111, projection=mapcrs)
ax.set_extent([-100, -10, 10, 70], ccrs.PlateCarree())    ## Can change the lat/lon bounds [West,East,South,North]

# Add geopolitical boundaries for map reference
ax.add_feature(cfeature.LAND, facecolor="#bdbdbd")
countries = NaturalEarthFeature(category="cultural", scale="110m", facecolor="none", name="admin_0_boundary_lines_land")
ax.add_feature(countries, linewidth=0.5, edgecolor="black")
ax.add_feature(cfeature.STATES.with_scale('50m'), linewidth=0.5)
ax.coastlines('50m', linewidth=0.8)

# Set up contour and fill intervals
hght_levels = arange(0,2100,30)   # (min,max,interval)
theta_levels = arange(240,330,3)

# Plot the variables
#thta_fcontours = plt.contourf(lons_2D, lats_2D, theta, levels=theta_levels, cmap=get_cmap('RdBu_r'), alpha=1, transform=ccrs.PlateCarree(),transform_first=True)
hght_contours = plt.contour(lons_2D, lats_2D, hght, levels=hght_levels, colors="black", linewidths=2.0, transform=ccrs.PlateCarree(),transform_first=True)
thta_contours = plt.contour(lons_2D, lats_2D, theta, levels=theta_levels, colors="red", linestyles='dashed', linewidths=2.0, transform=ccrs.PlateCarree(),transform_first=True)

# Plot 500-hPa wind barbs in knots, regrid to reduce number of barbs
uwndbrb = uwnd.values * units.meter/units.second
vwndbrb = vwnd.values * units.meter/units.second
ax.barbs(lons[::20], lats[::20], uwndbrb.to('kt').m[::20,::20], vwndbrb.to('kt').m[::20,::20], pivot='middle',     ## change the '20' to filter the spatial density of wind barbs #
         color='black', transform=datacrs)

#### Colorbar and contour labels #####
#cb = fig.colorbar(thta_fcontours, orientation='vertical', pad=0.03, extendrect=True, aspect=25, shrink=0.6)
#cb.set_label('Potential Temperature (K)', size='x-large')
plt.clabel(hght_contours, fmt='%d')
plt.clabel(thta_contours, fmt='%d')

##### Plot two titles, one on right and left side ######
plt.title('850-hPa Heights (m), Potential Temp. (K), and Wind Barbs (kts)', loc='left')
plt.title('Valid:'+startdate, loc='right')

##### Save figure #####
#fig.savefig('[insert_name].pdf')    # change file name here #